# Stereo Wind Retrieval — Debug Notebook

Walks through each pipeline stage with visual output for verification.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from stereo_winds.config import GOES16_CONFIG, GOES18_CONFIG, StereoPairConfig
from stereo_winds.navigation import (
    pixel_to_scanning_angle, scanning_angle_to_pixel,
    fixed_grid_to_geodetic, geodetic_to_fixed_grid,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Config & Data Loading

Load one timestep of GOES-16/GOES-18 Band 14 data in native grid.

In [ ]:
from stereo_winds.data_loading import load_stereo_scenes

t0 = datetime(2024, 1, 15, 12, 0)
# scenes = load_stereo_scenes(
#     t0=t0, dt_minutes=10.0,
#     sat_a_id='goes16', sat_b_id='goes18',
#     band_a='C14', cache_dir='../cache'
# )
# a0_data, sat_a = scenes['A0']
# b_minus_data, sat_b = scenes['B_minus']

# For offline testing, create synthetic data:
sat_a = GOES16_CONFIG
sat_b = GOES18_CONFIG
n = 256  # small grid for testing

from stereo_winds.config import SatelliteConfig
sat_a_small = SatelliteConfig(
    satellite_id='goes16', sub_lon_deg=-75.0,
    scale_x=sat_a.scale_x * (5424 / n), scale_y=sat_a.scale_y * (5424 / n),
    x_offset=sat_a.x_offset, y_offset=sat_a.y_offset,
    n_rows=n, n_cols=n,
)
sat_b_small = SatelliteConfig(
    satellite_id='goes18', sub_lon_deg=-137.0,
    scale_x=sat_b.scale_x * (5424 / n), scale_y=sat_b.scale_y * (5424 / n),
    x_offset=sat_b.x_offset, y_offset=sat_b.y_offset,
    n_rows=n, n_cols=n,
)

# Synthetic image (smooth gradient + noise)
rng = np.random.default_rng(42)
a0_data = np.sin(np.linspace(0, 4*np.pi, n))[:, None] * np.cos(np.linspace(0, 4*np.pi, n))[None, :]
a0_data = (a0_data + 1) * 128 + rng.normal(0, 5, (n, n))
a0_data = a0_data.astype(np.float32)

print(f'Image shape: {a0_data.shape}')
print(f'Sat A: {sat_a_small.satellite_id} at {sat_a_small.sub_lon_deg}°')
print(f'Sat B: {sat_b_small.satellite_id} at {sat_b_small.sub_lon_deg}°')

## 2. Remap Verification

Build LUT, remap B to A's grid. Verify geographic alignment.

In [ ]:
from stereo_winds.remap import build_remap_lut, remap_image, compute_valid_mask

col_b, row_b = build_remap_lut(sat_a_small, sat_b_small)
valid_mask = compute_valid_mask(col_b, row_b, sat_a_small, sat_b_small)

print(f'LUT shape: {col_b.shape}')
print(f'Valid overlap: {valid_mask.sum()} / {valid_mask.size} pixels ({100*valid_mask.sum()/valid_mask.size:.1f}%)')

# Create a synthetic B image and remap it
b_data = a0_data + rng.normal(0, 3, a0_data.shape).astype(np.float32)
b_remapped = remap_image(b_data, col_b, row_b)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(a0_data, cmap='gray')
axes[0].set_title('A0')
axes[1].imshow(b_data, cmap='gray')
axes[1].set_title('B original')
axes[2].imshow(b_remapped, cmap='gray')
axes[2].set_title('B remapped to A grid')
plt.tight_layout()

## 3-6. Disparity Fields

With real data, run RAFT on all 4 pairs. Here we use synthetic disparities.

In [ ]:
from stereo_winds.solver import compute_parallax_vectors, build_design_matrix, solve_stereo_winds

# Compute parallax vectors
w_u, w_v = compute_parallax_vectors(sat_a_small, sat_b_small)

from stereo_winds.visualize import plot_parallax_vectors
fig = plot_parallax_vectors(w_u, w_v, sat_a_small, step=16)
plt.show()

## 7-8. Parallax Isolation & Solver

Generate synthetic disparities with known truth and run the solver.

In [ ]:
# True state: h=8000m, V_u=3 px/dt, V_v=-2 px/dt, p_u=0.2, p_v=-0.1
h_true = 8000.0
V_u_true, V_v_true = 3.0, -2.0
p_u_true, p_v_true = 0.2, -0.1

dt_am, dt_ap, dt_bm, dt_bp = -600.0, 600.0, -600.0, 600.0
H_mat = build_design_matrix(w_u, w_v, dt_am, dt_ap, dt_bm, dt_bp)

x_true = np.array([h_true, p_u_true, p_v_true, V_u_true, V_v_true])
y_synth = np.einsum('...ij,...j->...i', H_mat, x_true)

# Add noise
y_synth += rng.normal(0, 0.3, y_synth.shape)

disparities = {
    'D1': np.stack([y_synth[:,:,0], y_synth[:,:,1]]),
    'D2': np.stack([y_synth[:,:,2], y_synth[:,:,3]]),
    'D3': np.stack([y_synth[:,:,4], y_synth[:,:,5]]),
    'D4': np.stack([y_synth[:,:,6], y_synth[:,:,7]]),
}

solution = solve_stereo_winds(disparities, H_mat)

print(f'Height: true={h_true:.0f}m, recovered={np.nanmean(solution["h"]):.0f}m')
print(f'V_u: true={V_u_true:.1f}, recovered={np.nanmean(solution["V_u"]):.1f}')
print(f'V_v: true={V_v_true:.1f}, recovered={np.nanmean(solution["V_v"]):.1f}')
print(f'p_u: true={p_u_true:.2f}, recovered={np.nanmean(solution["p_u"]):.2f}')
print(f'Chi2: mean={np.nanmean(solution["chi2"]):.2f}')

## 9. Solver Output Visualization

In [ ]:
from stereo_winds.solver import pixels_to_wind_ms

u_ms, v_ms = pixels_to_wind_ms(solution['V_u'], solution['V_v'], sat_a_small, 600.0)
solution['u_wind'] = u_ms
solution['v_wind'] = v_ms

from stereo_winds.visualize import plot_height_histogram
fig = plot_height_histogram(solution['h'], solution['quality_flag'])
plt.show()

## 10. Sanity Checks

Ground point statistics and quality assessment.

In [ ]:
from stereo_winds.validation.ground_points import ground_point_statistics

stats = ground_point_statistics(
    solution['h'], solution['u_wind'], solution['v_wind'],
    solution['quality_flag'],
)
print('Ground point statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

# Quality summary
qf = solution['quality_flag']
print(f'\nQuality: {(qf > 0).sum()}/{qf.size} pixels ({100*(qf > 0).sum()/qf.size:.1f}%)')